In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import userdata
token =userdata.get('Cryptonite-Token')

In [3]:
!git config --global user.email "idantsri2007@gmail.com"
!git config --global user.name "SILETRO"

In [4]:
!git clone https://{token}@github.com/SILETRO/Cryptonite-JM-Tasks.git
%cd Cryptonite-JM-Tasks

Cloning into 'Cryptonite-JM-Tasks'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 44 (delta 7), reused 14 (delta 4), pack-reused 17 (from 1)
Receiving objects: 100% (44/44), 56.11 MiB | 15.30 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/Cryptonite-JM-Tasks


In [13]:
!cp '/content/drive/MyDrive/Colab Notebooks/GenAI.ipynb' Trainee-Task0/GenAI/

In [19]:
!git add .

In [20]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [17]:
!git commit -m "Modified notebook"
!git push -u origin main

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
To https://github.com/SILETRO/Cryptonite-JM-Tasks.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/SILETRO/Cryptonite-JM-Tasks.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer
import nltk

In [ ]:
nltk.download('punkt_tab')
file_path = "/content/AllCombined 2.txt"

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
def load_and_chunk_text(file_path, chunk_size=500, overlap=50):
    with open(file_path, 'r', encoding='utf-8') as f:
        full_text = f.read()

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    tokens = tokenizer.encode(full_text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]

        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)

        start += (chunk_size - overlap)

    print(f"Total chunks created: {len(chunks)}")
    return chunks

text_chunks = load_and_chunk_text(file_path)

In [ ]:
import sys
!apt install -y libomp-dev
!{sys.executable} -m pip install faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np

def create_vector_store(chunks):
    embedder = SentenceTransformer('all-MiniLM-L6-v2')

    embeddings = embedder.encode(chunks, convert_to_numpy=True)
    embeddings = embeddings.astype('float32')

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)

    index.add(embeddings)

    print(f"Index built with {index.ntotal} vectors.")
    return index, embedder

vector_index, embed_model = create_vector_store(text_chunks)

In [ ]:
def retrieve_context(query, index, chunks, model, k=3):

    query_vector = model.encode([query], convert_to_numpy=True).astype('float32')
    distances, indices = index.search(query_vector, k)

    retrieved_chunks = [chunks[idx] for idx in indices[0]]

    if distances[0][0] > 1.5:
        return None

    return retrieved_chunks

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def load_generator_model():
    model_name = "google/flan-t5-small" # 80M params

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, from_tf=True)

    return tokenizer, model

gen_tokenizer, gen_model = load_generator_model()

In [ ]:
import re
from nltk.tokenize import sent_tokenize

def post_process_answer(answer_text):

    cleaned = re.sub(r'\([^)]*\)', '', answer_text)

    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    sentences = sent_tokenize(cleaned)
    final_sentences = []
    buffer_sent = ""

    for sent in sentences:
        if len(sent.split()) < 5:
            buffer_sent += " " + sent
        else:
            if buffer_sent:
                final_sentences.append(buffer_sent.strip() + " " + sent)
                buffer_sent = ""
            else:
                final_sentences.append(sent)

    if buffer_sent and final_sentences:
        final_sentences[-1] += " " + buffer_sent
    elif buffer_sent:
        final_sentences.append(buffer_sent)

    return " ".join(final_sentences[:4])

In [ ]:
def answer_question(question):
    print(f"\nUser Question: {question}")

    context_chunks = retrieve_context(question, vector_index, text_chunks, embed_model)

    if not context_chunks:
        return "Not enough information in the Simple Wikipedia dataset."

    context_text = " ".join(context_chunks)

    input_text = f"""You are an expert assistant. Using ONLY the context below, provide a detailed, complete, and well-explained answer.
                If needed, include multiple paragraphs. Context: {context_text} Question:{question} Answer: """

    inputs = gen_tokenizer(input_text, return_tensors="pt",max_length=10000, truncation=True)

    outputs = gen_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=800,
        do_sample=False,
        num_beams = 3,
        length_penalty =1.1,
        temperature=0.5
    )
    raw_answer = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)

    final_answer = post_process_answer(raw_answer)

    return final_answer

In [ ]:
response = answer_question("What is photosynthesis?")
print(f"Agent Answer: {response}")


User Question: What is photosynthesis?


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Agent Answer: oxygen gas diffuses out of the plant as a waste product of photosynthetic reactions, and atp is synthesized from adp and inorganic phosphate. this all happens in the grana of chloroplasts. during this reaction, sugars are built up using carbon dioxide and the products of the light - dependent reactions and various other chemicals found in the plant in the calvin cycle. therefore, the light - independent reaction cannot happen without the light - dependent reaction.


In [ ]:
answer_question("What did Edison do?")


User Question: What did Edison do?


'Inventor and entrepreneur, who invented many kinds of inventions and inventions. he developed one of the first practical light bulbs, but contrary to popular belief did not invent the light bulb. he started the general electric company to make some of the things he invented. edison developed one of the first practical light bulbs, but contrary to popular belief did not invent the light bulb.'

In [ ]:
answer_question("Who is Donald Trump")


User Question: Who is Donald Trump


'president of the united states in the 2024 presidential election. He was acquitted by the senate in february 2022, making him the third president in american history to be impeached twice. in january 2021, trump controversially made a telephone call with georgia secretary of state brad raffensperger. in the call, he was reported to have tried to change the election results.'